In [1]:
import logging
import claytonlib as clayton
from claytonlib.chart import (
    ChartSafariInput,
    chart_safari,
    evaluate_chart,
    evaluate_chart_top_10,
    chart_config,
    STRATEGY_ONLY_BALLS,
    STRATEGY_ONE_MUD,
    STRATEGY_SIX_BAIT,
    CRITERIA_CAPTURE,
    CRITERIA_WONT_FLEE_10_TURNS,
    CRITERIA_CAPTURE_MACHETE_AFTER_5_BALLS,
    SlidingWindowSum,
    NormalWindow,
)
from claytonlib.safari import safari_pokemon_by_name
from claytonlib.compass import (
    CompassSafariInput,
    compass_safari,
    compass_config,
)
from claytonlib.machete import (
    machete_one,
    machete_all,
    machete_jane,
    machete_config,
    JaneNode,
)

In [5]:
# --- Logging ---
# INFO shows per-write-cycle timing; DEBUG adds per-turn RNG detail
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s %(levelname)s %(name)s: %(message)s',
    datefmt='%H:%M:%S',
)
logging.getLogger('claytonlib.machete').setLevel(logging.DEBUG)

# --- Target ---
POKEMON_NAME = 'metang'
KEY_SEED     = 0xEC1504DC # delay = 1244
setup_delay_seconds = 300 # 5 minutes of setup
max_target_seconds = 1800 # Just one minute for testing

pokemon  = safari_pokemon_by_name(POKEMON_NAME)
strategy = STRATEGY_SIX_BAIT
#criteria = clayton.chart.machete_x_turns_n_balls_criteria(t50, 5)
# strategy = STRATEGY_SIX_BAIT
criteria = clayton.chart.machete_x_turns_n_balls_criteria(50, 5)

# --- Chart inputs ---
# setup_delay_seconds: real-world seconds between hitting the key seed and
# the first reachable encounter (seed confirmation, RNG advance, Sweet Scent, etc.)
# max_target_seconds: last second offset to evaluate (defines the chart window)
inputs = ChartSafariInput(
    key_seed             = KEY_SEED,
    setup_delay_seconds  = setup_delay_seconds,
    max_target_seconds   = max_target_seconds,
    strategy             = strategy,
    criteria             = criteria,
    pokemon              = pokemon,
)

# --- chart_config options ---
chart_config().evaluation_frames_per_write_cycle = 10
chart_config().resume_validation_enabled         = False
chart_config().evaluation_chains_per_write_cycle = 50
machete_config().max_turns                       = 12
# --- Evaluation strategies ---
# sigma_frames=12 ≈ 0.4 s timing window (1 frame = 2 delay units)
# eval_strategy = NormalWindow(sigma_frames=12)
eval_strategy = SlidingWindowSum(window=13)  # uniform 2-second window

# --- Compass inputs ---
# Set COMPASS_TARGET_DELAY to a delay value and COMPASS_INITIAL_TIME to the
# matching "Initial Time" from the evaluate_chart output above.
import datetime as dt
COMPASS_TARGET_DELAY = 48944 # 48948                          # e.g. 19232
COMPASS_INITIAL_TIME = dt.datetime(2000, 5, 27, 21, 57, 44)  # e.g. dt.datetime(2000, 1, 15, 12, 4, 42)
COMPASS_WINDOW       = 60                                     # search ±60 delay units (~1 second)

if COMPASS_TARGET_DELAY is not None and COMPASS_INITIAL_TIME is not None:
    compass_inputs = CompassSafariInput.from_chart(
        inputs,
        window        = COMPASS_WINDOW,
        initial_time  = COMPASS_INITIAL_TIME,
        target_delay  = COMPASS_TARGET_DELAY,
        # target_seed = 0x...  # optional: verified against target_delay + initial_time
        evaluation_strategy = STRATEGY_ONLY_BALLS,  # uncomment to show success column
    )

# --- Machete inputs ---
# MACHETE_SEED: the specific seed identified by compass (e.g. 0xCC16BF30)
# MACHETE_PATH: compass-syntax actions already observed before calling machete
#               (e.g. '010' if you threw ball×3 with 0/1/0 shakes). Leave '' if
#               starting from the beginning of the encounter.
# MACHETE_MAX_TURNS: override the global max_turns limit (None = unlimited)
MACHETE_SEED      = 0xCC16BF30  # e.g. from compass output
MACHETE_PATH      = ''          # e.g. '010' for actions already taken
MACHETE_MAX_TURNS = 20        # None = use machete_config() default (50)

In [6]:
clayton.evaluate_seed(KEY_SEED + 2, pokemon, strategy, criteria)

False

In [19]:
chart_safari(inputs)

11:28:40 INFO claytonlib.chart: Resuming incomplete chains from link 800.
11:28:40 INFO claytonlib.chart: Charting 2858 chain(s), links 800-1500.
11:29:02 INFO claytonlib.chart: links 800-809: eval=22.703s write=0.027s
11:29:19 INFO claytonlib.chart: links 810-819: eval=16.885s write=0.026s
11:29:38 INFO claytonlib.chart: links 820-829: eval=18.731s write=0.026s
11:29:55 INFO claytonlib.chart: links 830-839: eval=16.885s write=0.026s
11:30:19 INFO claytonlib.chart: links 840-849: eval=24.272s write=0.027s
11:30:43 INFO claytonlib.chart: links 850-859: eval=23.351s write=0.026s
11:31:01 INFO claytonlib.chart: links 860-869: eval=17.980s write=0.026s
11:31:21 INFO claytonlib.chart: links 870-879: eval=20.874s write=0.026s
11:31:43 INFO claytonlib.chart: links 880-889: eval=21.731s write=0.027s
11:32:06 INFO claytonlib.chart: links 890-899: eval=22.307s write=0.026s
11:32:27 INFO claytonlib.chart: links 900-909: eval=21.000s write=0.026s
11:32:47 INFO claytonlib.chart: links 910-919: eval

In [15]:
evaluate_chart_top_10(inputs, eval_strategy)

11:24:45 INFO claytonlib.chart: chain 2000-04-30_21-57-59+300.chain: flatten=0.006s score=0.003s results=0.034s total=0.044s
11:24:45 INFO claytonlib.chart: chain 2000-04-30_21-58-58+300.chain: flatten=0.006s score=0.003s results=0.035s total=0.044s
11:24:45 INFO claytonlib.chart: chain 2000-04-30_21-59-57+300.chain: flatten=0.006s score=0.003s results=0.032s total=0.042s
11:24:45 INFO claytonlib.chart: chain 2000-05-24_21-57-59+300.chain: flatten=0.006s score=0.003s results=0.035s total=0.045s
11:24:45 INFO claytonlib.chart: chain 2000-05-24_21-58-58+300.chain: flatten=0.006s score=0.003s results=0.035s total=0.044s
11:24:45 INFO claytonlib.chart: chain 2000-05-24_21-59-57+300.chain: flatten=0.006s score=0.004s results=0.034s total=0.043s
11:24:45 INFO claytonlib.chart: chain 2000-05-25_21-52-59+300.chain: flatten=0.006s score=0.004s results=0.035s total=0.045s
11:24:45 INFO claytonlib.chart: chain 2000-05-25_21-53-58+300.chain: flatten=0.006s score=0.003s results=0.035s total=0.044s


Top 10 (by score)
 #        Score(p)    Delay      Time        Δ Time  Initial Time
-----------------------------------------------------------------
 1     7.97(61.3%)    90172  22:22:41   24m 42.133s  2000-04-30 21:57:59
 2     7.87(60.5%)    23568  21:59:59    6m 12.067s  2000-08-17 21:53:47
 3     7.77(59.8%)    23566  21:59:59    6m 12.033s  2000-08-17 21:53:47
 4     7.73(59.5%)    65380  21:53:59   17m 48.933s  2000-07-27 21:36:11
 5      7.7(59.2%)    23568  21:55:59    6m 12.067s  2000-05-28 21:49:47
 6      7.7(59.2%)    81114  21:56:58   22m 11.167s  2000-05-31 21:34:47
 7     7.67(59.0%)    27958  22:04:59    7m 25.233s  2000-05-29 21:57:34
 8     7.63(58.7%)    81114  21:55:59   22m 11.167s  2000-05-31 21:33:48
 9      7.6(58.5%)    23566  21:55:59    6m 12.033s  2000-05-28 21:49:47
10     7.57(58.2%)    23566  21:56:58    6m 12.033s  2000-05-28 21:50:46

Best 10 (highest score at each successively lower delay)
 #        Score(p)    Delay      Time        Δ Time  Initial T

In [ ]:
print(input("Input"))

In [11]:
compass_safari(compass_inputs)

=== Compass: Safari Zone Seed Identifier ===
  m      Mud, no crit         Metang is angry!
  M / a  Mud, crit (Anger)    Metang is beside itself with anger!
  b      Bait, no crit        Metang is eating!
  B / e  Bait, crit (Eating)  Metang is busy eating!
  0      Ball, 0 shakes       Oh, no! The Pokémon broke free!
  1      Ball, 1 shake        Aww! It appeared to be caught!
  2      Ball, 2 shakes       Aargh! Almost had it!
  3      Ball, 3 shakes       Shoot! It was so close, too!
  C      Captured (ends)      Gotcha! Metang was caught!
  F      Fled (ends)          Metang fled!
  u      Undo last action     —
  ?x     Uncertain result     —
  Spaces and commas in input are ignored.


Seeds: 119 / 119 remaining
Path:  (none)
Balls: 30
   #        Seed    Delay      Δ
   1. 0xCB16BEF4    48884    -60
   2. 0xCB16BEF6    48886    -58
   3. 0xCC16BEF6    48886    -58
   4. 0xCB16BEF8    48888    -56
   5. 0xCC16BEF8    48888    -56



>>  1m0



Seeds: 11 / 119 remaining
Path:  1m0
Balls: 28
   #        Seed    Delay      Δ
   1. 0xCB16BEFA    48890    -54
   2. 0xCC16BEFC    48892    -52
   3. 0xCC16BF1C    48924    -20
   4. 0xCB16BF20    48928    -16
   5. 0xCC16BF20    48928    -16



>>  b1Bm



Seeds: 1 / 119 remaining
Path:  1m0b1Bm
Balls: 27
   #        Seed    Delay      Δ  Success
   1. 0xCC16BF32    48946     +2  no

╔════════════════════╗
║  Seed identified!  ║
║  seed  = 0xCC16BF32║
║  delay = 48946     ║
║  Δ     = +2        ║
║  path  = 1m0b1Bm   ║
╚════════════════════╝


In [6]:
# machete_one — find the shortest capture path for the identified seed
result = machete_one(pokemon, seed=0xCC16BEFC, path=MACHETE_PATH, max_turns=24)
if result is None:
    print("No capture path found within the turn limit.")
else:
    print(f"Shortest path: {result}  ({len(result)} actions)")

Shortest path: mmbm10Bbbm1m10bB21mmC  (21 actions)


In [4]:
# machete_all — find every capture path for the identified seed
all_paths, truncated = machete_all(pokemon, seed=0xCC16BEFC, path=MACHETE_PATH, max_turns=19)
print(f"{len(all_paths)} capture path(s) found, {truncated} branch(es) truncated by turn limit.")
for p in all_paths[:20]:  # show first 20 to avoid flooding output
    print(f"  {p}")
if len(all_paths) > 20:
    print(f"  ... ({len(all_paths) - 20} more)")

08:56:13 DEBUG claytonlib.machete: machete_all: 500000 nodes  0 paths  798468 truncated  1.469s  (340k nodes/s)
08:56:15 DEBUG claytonlib.machete: machete_all: 1000000 nodes  0 paths  1625136 truncated  2.953s  (339k nodes/s)
08:56:16 DEBUG claytonlib.machete: machete_all: 1500000 nodes  0 paths  2453328 truncated  4.420s  (339k nodes/s)
08:56:18 DEBUG claytonlib.machete: machete_all: 2000000 nodes  0 paths  3283809 truncated  5.892s  (339k nodes/s)
08:56:19 DEBUG claytonlib.machete: machete_all: 2500000 nodes  0 paths  4130184 truncated  7.365s  (339k nodes/s)
08:56:21 DEBUG claytonlib.machete: machete_all: 3000000 nodes  0 paths  5008947 truncated  8.855s  (339k nodes/s)
08:56:22 DEBUG claytonlib.machete: machete_all: 3500000 nodes  0 paths  5823918 truncated  10.306s  (340k nodes/s)
08:56:24 DEBUG claytonlib.machete: machete_all: 4000000 nodes  0 paths  6644922 truncated  11.770s  (340k nodes/s)
08:56:25 DEBUG claytonlib.machete: machete_all: 4500000 nodes  0 paths  7476267 truncate

KeyboardInterrupt: 

In [20]:
# machete_jane — optimal decision tree across all compass candidates
# Requires compass_inputs to be configured above and compass to have narrowed candidates.
# Use _generate_candidates to pull the full compass window, or pass plain seed ints directly.
from claytonlib.compass import _generate_candidates  # internal, for notebook use

compass_candidates = [
    (ctx, seed)
    for ctx, seed, delay in _generate_candidates(compass_inputs)
]
# Alternatively, pass plain seed ints with pokemon:
compass_candidates = [0xCC16BF30, 0xCC16BF32, 0xCC16BEFC]
tree = machete_jane(compass_candidates, pokemon=pokemon, max_turns=18)

# tree = machete_jane(compass_candidates, max_turns=10, interactive=True)


def print_tree(node, indent=0, outcome=None):
    prefix = "  " * indent
    label = f"[{outcome}] " if outcome else ""
    if node is None:
        print(f"{prefix}{label}(none)")
        return
    prob_pct = float(node.probability) * 100
    cap_str = ""
    if node.direct_capture_prob is not None:
        cap_str = f"  capture={float(node.direct_capture_prob)*100:.1f}%"
    print(f"{prefix}{label}{node.action}  p={prob_pct:.1f}%{cap_str}")
    if node.branches:
        for ch, child in sorted(node.branches.items()):
            print_tree(child, indent + 1, outcome=ch)


print_tree(tree)
# print(compass_candidates)

13:08:19 INFO claytonlib.machete: machete_jane: 3 candidate(s), max_turns=18
13:08:19 DEBUG claytonlib.machete: machete_jane: candidate 1/3
13:08:20 DEBUG claytonlib.machete: machete_all done: 18 path(s), 264627 truncated, elapsed=0.471s
13:08:20 DEBUG claytonlib.machete: machete_jane: candidate 2/3
13:08:21 DEBUG claytonlib.machete: machete_all: 500000 nodes  861 paths  767505 truncated  1.481s  (338k nodes/s)
13:08:23 DEBUG claytonlib.machete: machete_all: 1000000 nodes  1398 paths  1563054 truncated  2.930s  (341k nodes/s)
13:08:24 DEBUG claytonlib.machete: machete_all: 1500000 nodes  1834 paths  2366310 truncated  4.378s  (343k nodes/s)
13:08:25 DEBUG claytonlib.machete: machete_all done: 1870 path(s), 2942238 truncated, elapsed=5.385s
13:08:25 DEBUG claytonlib.machete: machete_jane: candidate 3/3
13:08:27 DEBUG claytonlib.machete: machete_all: 500000 nodes  0 paths  772200 truncated  1.449s  (345k nodes/s)
13:08:28 DEBUG claytonlib.machete: machete_all: 1000000 nodes  0 paths  156

BALL  p=66.7%
  [0] BALL  p=100.0%
    [1] BALL  p=100.0%
      [0] BALL  p=100.0%
        [0] BALL  p=100.0%
          [2] BALL  p=100.0%  capture=100.0%
            [C] CAPTURED  p=100.0%
  [1] MUD  p=50.0%
    [m] BALL  p=50.0%
      [0] BAIT  p=50.0%
        [b] BALL  p=100.0%
          [1] BAIT  p=100.0%
            [B] BALL  p=100.0%
              [0] BALL  p=100.0%
                [1] MUD  p=100.0%
                  [m] BAIT  p=100.0%
                    [b] BALL  p=100.0%
                      [0] BALL  p=100.0%
                        [1] MUD  p=100.0%
                          [m] BALL  p=100.0%
                            [1] MUD  p=100.0%
                              [m] MUD  p=100.0%
                                [m] BALL  p=100.0%  capture=100.0%
                                  [C] CAPTURED  p=100.0%


In [7]:
from claytonlib.compass_premetronome import (
    analyze_compass_premetronome,
    CompassPremetronomeInput,
    MetronomeOpponent,
)
from collections import Counter

metronome_analysis_inputs = CompassPremetronomeInput(
    opponent     = MetronomeOpponent.MAGIKARP,
    key_seed     = KEY_SEED,
    target_delay = COMPASS_TARGET_DELAY,
    initial_time = COMPASS_INITIAL_TIME,
    window       = 120,
)

results = analyze_compass_premetronome(metronome_analysis_inputs)

counts = Counter(move for _, move in results)
print(f"{len(results)} candidates  |  {len(counts)} distinct moves\n")
print(f"  {'Move':<24}  {'Count':>5}  {'%':>6}")
print(f"  {'-'*24}  {'-----':>5}  {'------':>6}")
counter_counter = Counter()
for move, count in counts.most_common():
    counter_counter.update({count, 1})
    print(f"  {move:<24}  {count:>5}  {count/len(results)*100:>5.1f}%")
print("Frequency of moves with given frequency")
for move, count in counter_counter.most_common():
    print(f"  {move:<24}  {count:>5}  {count/len(results)*100:>5.1f}%")


237 candidates  |  195 distinct moves

  Move                      Count       %
  ------------------------  -----  ------
  Shadow Ball                   3    1.3%
  Dragon Dance                  3    1.3%
  Wake Up Slap                  3    1.3%
  Spikes                        2    0.8%
  Weather Ball                  2    0.8%
  Fissure                       2    0.8%
  Safeguard                     2    0.8%
  Fury Cutter                   2    0.8%
  Leaf Storm                    2    0.8%
  Leer                          2    0.8%
  Wing Attack                   2    0.8%
  Skill Swap                    2    0.8%
  Poison Sting                  2    0.8%
  Twineedle                     2    0.8%
  Haze                          2    0.8%
  Recover                       2    0.8%
  Vine Whip                     2    0.8%
  Razor Wind                    2    0.8%
  Poison Jab                    2    0.8%
  Aqua Jet                      2    0.8%
  Worry Seed                    2    